# Speed of Light Model Metric Calculations

Apply model metrics to Speed of Light formulas to get actual values.

In [143]:
USE_PLAYROOM = True

# Garden metrics.
W = 1_297
H = 840
T = 4_256
P = 4_140_699
V = 1_438_668
N = 3_085_575
S = 53

# Playroom metrics.
if USE_PLAYROOM:
    W = 1_264
    H = 832
    T = 4_108
    P = 1_908_841
    V = 244_504
    N = 1_298_258
    S = 90


## Device metrics

In [144]:
MAX_BANDWIDTH_BYTES = 1_008 * 1e9
MAX_FLOPS = 82.6 * 1e12

## Formulas

In [145]:
def kernel_time(kernel_io: float, kernel_flop: float) -> float:
    return (kernel_io / MAX_BANDWIDTH_BYTES + kernel_flop / MAX_FLOPS) * 1e6

### Preprocess

In [146]:
preprocess_io = 236 * P + 56 * V
preprocess_flop = 6 * P + 365 * V + 44 * N
preprocess_time = kernel_time(preprocess_io, preprocess_flop)
print(f"{preprocess_io:,}")
print(f"{preprocess_flop:,}")
print(f"{preprocess_time:,}")

464,178,700
157,820,358
462.40540015065915


### Depth Sort

In [147]:
depth_sort_io = 192 * V
depth_sort_time = kernel_time(depth_sort_io, 0)
print(f"{depth_sort_io:,}")
print(f"{depth_sort_time:,}")

46,944,768
46.57219047619048


### Apply Depth Ordering

In [148]:
apply_depth_order_io = 12 * V
print(f"{apply_depth_order_io:,}")

2,934,048


### Exclusive Sum

In [149]:
exclusive_sum_io = 8 * V
print(f"{exclusive_sum_io:,}")

1,956,032


### Create Instances

In [150]:
create_instances_io = 36 * V + 6 * N
create_instances_flop = 4 * V + 44 * N
create_instances_time = kernel_time(create_instances_io, 0)
print(f"{create_instances_io:,}")
print(f"{create_instances_flop:,}")
print(f"{create_instances_time:,}")

16,591,692
58,101,368
16.460011904761902


### Group by Tile (Sort)

In [151]:
group_by_tile_io = 72 * V
group_by_tile_time = kernel_time(group_by_tile_io, 0)
print(f"{group_by_tile_io:,}")
print(f"{group_by_tile_time:,}")

17,604,288
17.46457142857143


### Extract Instance Ranges

In [152]:
extract_instance_ranges_io = 2 * N + 8 * T
print(f"{extract_instance_ranges_io:,}")

2,629,380


### Blend

In [153]:
blend_io = 8 * T + 44 * N + 12 * W * H
blend_flop = (8 + 18 * S) * W * H
blend_time = kernel_time(blend_io, 0)
print(f"{blend_io:,}")
print(f"{blend_flop:,}")
print(f"{blend_time:,}")

69,775,992
1,712,082,944
69.22221428571429


## Combined

In [154]:
io = preprocess_io + depth_sort_io + apply_depth_order_io + exclusive_sum_io + create_instances_io + group_by_tile_io + extract_instance_ranges_io + blend_io
flop = preprocess_flop + create_instances_flop + blend_flop

assert io == 236 * P + 376 * V + 52 * N + 16 * T + 12 * W * H
assert flop == 6 * P + 369 * V + 88 * N + (8 + 18 * S) * W * H

print(f"{io:,}")
print(f"{flop:,}")

622,614,900
1,928,004,670


## Time and FPS

In [155]:
time = kernel_time(io, flop)
fps = 1 / (time * 1e-6)
print(f"{time}")
print(f"{fps:,}")

641.0149728006456
1,560.0259626244301
